# Advanced Retrieval Techniques

This notebook demonstrates advanced retrieval techniques for RAG: Query Rewriting, HyDE, and Cross-Encoder Reranking.

## Techniques Covered

1. **Query Rewriting** - Transform queries to improve retrieval
2. **HyDE** - Hypothetical Document Embeddings
3. **Cross-Encoder Reranking** - Improve result quality with second-stage reranking

## Prerequisites

Install Ollama and pull the required models:
```bash
ollama pull llama3.2
ollama pull nomic-embed-text
```

In [ ]:
# Install dependencies
!pip install langchain langchain-community langchain-ollama chromadb sentence-transformers

In [ ]:
import os
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from sentence_transformers import CrossEncoder

## 1. Setup Documents

In [ ]:
# Sample documents about RAG
documents = [
    Document(
        page_content="""Retrieval-Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by retrieving relevant information from external knowledge bases. This helps the model generate more accurate and up-to-date responses.""",
        metadata={"source": "doc1", "topic": "intro"}
    ),
    Document(
        page_content="""The basic RAG pipeline consists of three components: 1) Retriever that finds relevant documents using embeddings and vector similarity search, 2) Database storing document embeddings, and 3) Generator that produces final output using retrieved context.""",
        metadata={"source": "doc2", "topic": "architecture"}
    ),
    Document(
        page_content="""Embedding models convert text into numerical vectors (embeddings) that capture semantic meaning. Popular embedding models include OpenAI's text-embedding-3-small, Cohere, and open-source options like nomic-embed-text.""",
        metadata={"source": "doc3", "topic": "embeddings"}
    ),
    Document(
        page_content="""Vector databases like Chroma, Pinecone, Weaviate, and Milvus store embeddings and enable fast similarity search. They typically use algorithms like HNSW or IVF for efficient nearest neighbor search.""",
        metadata={"source": "doc4", "topic": "vector-db"}
    ),
    Document(
        page_content="""HyDE (Hypothetical Document Embeddings) is a technique where you generate a hypothetical answer document and use it for retrieval. This helps bridge the gap between query language and document language.""",
        metadata={"source": "doc5", "topic": "hyde"}
    ),
    Document(
        page_content="""Query rewriting improves retrieval by transforming user queries before searching. This helps when users ask questions in ways that don't directly match how information is stored in the knowledge base.""",
        metadata={"source": "doc6", "topic": "query-rewrite"}
    ),
    Document(
        page_content="""Reranking is a second-stage retrieval technique that improves result quality. After initial retrieval, a more sophisticated model re-scores and re-orders the results for better relevance.""",
        metadata={"source": "doc7", "topic": "reranking"}
    ),
    Document(
        page_content="""Self-RAG is a framework where the LLM learns to decide when to retrieve information and how to evaluate its own outputs for relevance and accuracy.""",
        metadata={"source": "doc8", "topic": "self-rag"}
    )
]

print(f"Loaded {len(documents)} documents")

In [ ]:
# Create vector store
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="advanced_rag_demo"
)

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Vector store created!")

## 2. Baseline Retrieval (No Advanced Techniques)

In [ ]:
def display_results(query, results, title="Results"):
    """Display retrieval results."""
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"\n{title}:")
    print("-" * 60)
    for i, doc in enumerate(results, 1):
        print(f"\n{i}. [{doc.metadata.get('source', 'unknown')}] {doc.page_content[:150]}...")

# Test baseline
query = "How do I improve my RAG system?"
baseline_results = base_retriever.get_relevant_documents(query)
display_results(query, baseline_results, "Baseline Retrieval")

## 3. Query Rewriting

In [ ]:
# Initialize LLM
llm = ChatOllama(model="llama3.2")

In [ ]:
class QueryRewriter:
    """Rewrite queries for better retrieval."""
    
    def __init__(self, llm):
        self.llm = llm
    
    def rewrite(self, query: str) -> str:
        """Rewrite a single query."""
        prompt = f"""Rewrite this query to improve document retrieval.
        Keep it as a question, but use different words and make it more explicit.

Original: {query}

Rewritten:"""
        response = self.llm.invoke(prompt)
        return response.content.strip()
    
    def multi_query(self, query: str) -> list:
        """Generate multiple query variations."""
        prompt = f"""Generate 3 different versions of this query that might
        retrieve relevant documents. Consider different phrasings and synonyms.

Original: {query}

Queries:"""
        response = self.llm.invoke(prompt)
        
        # Parse response
        lines = response.content.strip().split('\n')
        queries = [query]  # Include original
        for line in lines:
            line = line.strip()
            if line and (line[0].isdigit() or line.startswith('-')):
                q = line.lstrip('0123456789.-').strip()
                if q:
                    queries.append(q)
        return queries[:4]

In [ ]:
# Test query rewriting
rewriter = QueryRewriter(llm)

query = "How do I improve my RAG system?"
rewritten = rewriter.rewrite(query)
print(f"Original: {query}")
print(f"Rewritten: {rewritten}")

In [ ]:
# Test multi-query
multi_queries = rewriter.multi_query(query)
print(f"\nMultiple queries generated:")
for i, q in enumerate(multi_queries, 1):
    print(f"  {i}. {q}")

In [ ]:
# Multi-query retrieval
class MultiQueryRetriever:
    """Retrieve using multiple query variations."""
    
    def __init__(self, base_retriever, llm):
        self.base_retriever = base_retriever
        self.rewriter = QueryRewriter(llm)
    
    def retrieve(self, query: str, k: int = 4) -> list:
        """Retrieve with multiple queries."""
        queries = self.rewriter.multi_query(query)
        
        all_docs = []
        for q in queries:
            docs = self.base_retriever.get_relevant_documents(q)
            all_docs.extend(docs)
        
        # Deduplicate
        seen = set()
        unique = []
        for doc in all_docs:
            key = doc.page_content[:50]
            if key not in seen:
                seen.add(key)
                unique.append(doc)
        
        return unique[:k]

In [ ]:
# Test with query rewriting
mq_retriever = MultiQueryRetriever(base_retriever, llm)
query_rewrite_results = mq_retriever.retrieve(query)
display_results(query, query_rewrite_results, "With Query Rewriting")

## 4. HyDE (Hypothetical Document Embeddings)

In [ ]:
class HyDERetriever:
    """HyDE: Hypothetical Document Embeddings."""
    
    def __init__(self, vectorstore, llm):
        self.vectorstore = vectorstore
        self.llm = llm
    
    def generate_hypothetical(self, query: str) -> str:
        """Generate a hypothetical answer document."""
        prompt = f"""Write a detailed hypothetical document that could answer this question.
        Include relevant facts and explanations.

Question: {query}

Hypothetical Document:"""
        response = self.llm.invoke(prompt)
        return response.content
    
    def retrieve(self, query: str, k: int = 4) -> list:
        """Retrieve using HyDE."""
        # Generate hypothetical document
        hypothetical = self.generate_hypothetical(query)
        print(f"Hypothetical document: {hypothetical[:200]}...")
        
        # Get embeddings
        query_emb = self.vectorstore.embedding.embed_query(query)
        hypo_emb = self.vectorstore.embedding.embed_query(hypothetical)
        
        # Combine embeddings (average)
        combined_emb = [(q + h) / 2 for q, h in zip(query_emb, hypo_emb)]
        
        # Search
        results = self.vectorstore.vectorstore.similarity_search_by_vector(
            combined_emb,
            k=k
        )
        
        return results

In [ ]:
# Test HyDE
hyde_retriever = HyDERetriever(vectorstore, llm)
hyde_results = hyde_retriever.retrieve(query)
display_results(query, hyde_results, "With HyDE")

## 5. Cross-Encoder Reranking

In [ ]:
# Initialize cross-encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

class CrossEncoderReranker:
    """Rerank using cross-encoder model."""
    
    def __init__(self, base_retriever, cross_encoder, top_k: int = 20):
        self.base_retriever = base_retriever
        self.cross_encoder = cross_encoder
        self.top_k = top_k
    
    def rerank(self, query: str, final_k: int = 4) -> list:
        """Rerank documents."""
        # Initial retrieval
        initial_docs = self.base_retriever.get_relevant_documents(query)
        initial_docs = initial_docs[:self.top_k]
        
        # Prepare pairs
        doc_contents = [doc.page_content for doc in initial_docs]
        pairs = [[query, doc] for doc in doc_contents]
        
        # Get scores
        scores = self.cross_encoder.predict(pairs)
        
        # Sort by score
        doc_scores = list(zip(initial_docs, scores))
        doc_scores.sort(key=lambda x: x[1], reverse=True)
        
        return doc_scores[:final_k]

In [ ]:
# Test reranking
reranker = CrossEncoderReranker(base_retriever, cross_encoder)
reranked_results = reranker.rerank(query)

print("\nResults with Cross-Encoder Reranking:")
print("=" * 60)
for i, (doc, score) in enumerate(reranked_results, 1):
    print(f"\n{i}. Score: {score:.4f}")
    print(f"   {doc.page_content[:150]}...")

## 6. Complete Advanced RAG Pipeline

In [ ]:
class AdvancedRAGPipeline:
    """Complete advanced RAG pipeline with all techniques."""
    
    def __init__(self, vectorstore, llm, cross_encoder):
        self.vectorstore = vectorstore
        self.llm = llm
        self.cross_encoder = cross_encoder
        
        self.base_retriever = vectorstore.as_retriever(search_kwargs={"k": 20})
        self.query_rewriter = QueryRewriter(llm)
        self.hyde = HyDERetriever(vectorstore, llm)
        self.reranker = CrossEncoderReranker(self.base_retriever, cross_encoder)
    
    def query(self, question: str, use_all_techniques: bool = True) -> dict:
        """Query the RAG system."""
        
        if use_all_techniques:
            # Step 1: Query rewriting
            rewritten = self.query_rewriter.rewrite(question)
            
            # Step 2: Multi-query retrieval
            mq_retriever = MultiQueryRetriever(self.base_retriever, self.llm)
            initial_docs = mq_retriever.retrieve(question, k=10)
            
            # Step 3: Reranking
            # (Combine with original query reranking)
            pairs = [[question, doc.page_content] for doc in initial_docs]
            scores = self.cross_encoder.predict(pairs)
            doc_scores = list(zip(initial_docs, scores))
            doc_scores.sort(key=lambda x: x[1], reverse=True)
            final_docs = [doc for doc, _ in doc_scores[:4]]
            
            return {
                "rewritten_query": rewritten,
                "documents": final_docs
            }
        else:
            # Baseline
            docs = self.base_retriever.get_relevant_documents(question)
            return {
                "rewritten_query": question,
                "documents": docs[:4]
            }

In [ ]:
# Test complete pipeline
advanced_rag = AdvancedRAGPipeline(vectorstore, llm, cross_encoder)

query = "How do I improve my RAG system?"
result = advanced_rag.query(query)

print(f"Original query: {query}")
print(f"Rewritten query: {result['rewritten_query']}")
print(f"\nFinal retrieved documents: {len(result['documents'])}")
for i, doc in enumerate(result['documents'], 1):
    print(f"\n{i}. {doc.page_content[:150]}...")

## 7. Comparison Summary

In [ ]:
# Compare all techniques
query = "How do I improve my RAG system?"

print("=" * 70)
print("COMPARISON: Advanced Retrieval Techniques")
print("=" * 70)

print("\n1. BASELINE RETRIEVAL")
baseline = base_retriever.get_relevant_documents(query)
print(f"   Retrieved {len(baseline)} documents")
print(f"   First result: {baseline[0].metadata.get('source', 'unknown')}")

print("\n2. QUERY REWRITING")
mq_result = mq_retriever.retrieve(query)
print(f"   Retrieved {len(mq_result)} unique documents")
print(f"   First result: {mq_result[0].metadata.get('source', 'unknown')}")

print("\n3. HYDE")
hyde_result = hyde_retriever.retrieve(query)
print(f"   Retrieved {len(hyde_result)} documents")
print(f"   First result: {hyde_result[0].metadata.get('source', 'unknown')}")

print("\n4. CROSS-ENCODER RERANKING")
reranked = reranker.rerank(query)
print(f"   Retrieved {len(reranked)} documents (from top 20)")
print(f"   First result: {reranked[0][0].metadata.get('source', 'unknown')}")

print("\n" + "=" * 70)

## Summary

This notebook covered three key advanced retrieval techniques:

1. **Query Rewriting**: Transform queries to better match document language
2. **HyDE**: Use hypothetical documents to improve semantic matching
3. **Cross-Encoder Reranking**: Second-stage re-scoring for better precision

## When to Use What

| Technique | Best For | Complexity |
|-----------|----------|------------|
| Query Rewriting | Most queries, especially with varied phrasing | Medium |
| HyDE | Conceptual/meaning-based queries | Medium |
| Reranking | When precision is critical | Low |
| Combined | Production systems | Higher |

## Additional Resources

- [HyDE Paper](https://arxiv.org/abs/2212.10496)
- [Query Rewriting Tutorial](https://dev.to/rogiia/build-an-advanced-rag-app-query-rewriting-h3p)
- [Cross-Encoder Documentation](https://sbert.net/examples/cross_encoder/training/rerankers/README.html)